# MIO-TCD inventory
**Mục tiêu:** kiểm kê annotation trước khi split. **Input:** `gt_train.csv`, ảnh raw (read-only). **Output:** `data/mio_tcd/metadata/inventory.csv`.

Assumption: `gt_train.csv` không có header; dataset được cấu hình qua `MIO_TCD_ROOT`, `data/mio_tcd/dataset_config.yaml`, hoặc override dưới đây.

## Config

In [6]:
DATASET_ROOT_OVERRIDE = Path(
    r"E:\anh thu\workspace\MIO-TCD-Localization\MIO-TCD-Localization"
)
FORCE_OVERWRITE = False

## Imports and validation

In [7]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT, resolve_dataset_root, read_annotations, build_inventory
DATASET_ROOT = resolve_dataset_root(DATASET_ROOT_OVERRIDE)
print(f'PROJECT_ROOT={PROJECT_ROOT}\nDATASET_ROOT={DATASET_ROOT}')

PROJECT_ROOT=E:\anh thu\workspace\area-violation-detection
DATASET_ROOT=E:\anh thu\workspace\MIO-TCD-Localization\MIO-TCD-Localization


## Load data and processing

In [8]:
annotations = read_annotations(DATASET_ROOT)
inventory = build_inventory(annotations, DATASET_ROOT, tqdm)
if inventory.empty: raise ValueError('Empty inventory')
print('Total annotations:', len(annotations), '| Unique images:', len(inventory))
display(annotations.original_class.value_counts().rename('annotations').to_frame())
display(annotations.mapped_class.fillna('ignored').value_counts().rename('annotations').to_frame())

Inspecting images: 100%|██████████| 110000/110000 [20:49<00:00, 88.05it/s] 


Total annotations: 351549 | Unique images: 110000


,annotations
original_class,
car,233497
pickup_truck,44283
motorized_vehicle,25845
bus,10598
articulated_truck,9301
work_van,8709
pedestrian,7128
single_unit_truck,5741
non-motorized_vehicle,2350


,annotations
mapped_class,
car,233497
truck,59325
ignored,36904
bus,10598
person,7128
bicycle,2260
motorcycle,1837


## Summary and save outputs

In [9]:
print('Objects/image summary'); display(inventory.num_objects.describe().to_frame().T)
print('Single-class images:', int((inventory[[c for c in inventory if c.startswith('has_')]].sum(axis=1) == 1).sum()))
print('Multi-class images:', int((inventory[[c for c in inventory if c.startswith('has_')]].sum(axis=1) > 1).sum()))
output = PROJECT_ROOT / 'data' / 'mio_tcd' / 'metadata' / 'inventory.csv'
if output.exists() and not FORCE_OVERWRITE: raise FileExistsError(f'{output} exists; set FORCE_OVERWRITE=True to replace it.')
output.parent.mkdir(parents=True, exist_ok=True); inventory.to_csv(output, index=False)
display(inventory.head()); print('Saved:', output)

Objects/image summary


,count,mean,std,min,25%,50%,75%,max
num_objects,110000.0,3.1959,2.927376,1.0,1.0,2.0,4.0,34.0


Single-class images: 68915
Multi-class images: 38164


,image_id,image_path,width,height,num_objects,num_target_objects,ignored_object_count,invalid_bbox_count,source_classes,valid,has_person,has_bicycle,has_car,has_motorcycle,has_bus,has_truck
0,00000000,E:\anh thu\workspace\MIO-TCD-Localization\MIO-...,342,228,5,5,0,0,articulated_truck;car;pickup_truck,True,False,False,True,False,False,True
1,00000001,E:\anh thu\workspace\MIO-TCD-Localization\MIO-...,720,480,6,6,0,0,bus;car,True,False,False,True,False,True,False
2,00000002,E:\anh thu\workspace\MIO-TCD-Localization\MIO-...,720,480,1,0,1,0,motorized_vehicle,True,False,False,False,False,False,False
3,00000003,E:\anh thu\workspace\MIO-TCD-Localization\MIO-...,720,480,2,2,0,0,car,True,False,False,True,False,False,False
4,00000004,E:\anh thu\workspace\MIO-TCD-Localization\MIO-...,720,480,1,1,0,0,bus,True,False,False,False,False,True,False


Saved: E:\anh thu\workspace\area-violation-detection\data\mio_tcd\metadata\inventory.csv
